In [1]:
!pip install -q sentence-transformers
!pip install -q datasets

In [2]:
!pip install -q --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.8/622.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.10.0 which is incompatible.


In [3]:
%cd /content/drive/MyDrive/auto-hh/src
%load_ext autoreload
%autoreload 2

/content/drive/MyDrive/auto-hh/src


In [13]:
import torch
from models import BiEncoder, CrossEncoder
from training import Trainer
from lib import create_pairs

In [ ]:
MODEL_NAME = "BAAI/bge-m3"

model = BiEncoder(
    model_name=MODEL_NAME,
    need_attention=True,
)

# Информация о модели
print(f"✅ Модель инициализирована")

In [ ]:
VACANCIES_PATH = "/content/drive/MyDrive/auto-hh/hh_dataset.csv"
RESUMES_PATH = "/content/drive/MyDrive/auto-hh/resumes_generated.csv"

train_dataset = create_pairs(
    vacancies_path=VACANCIES_PATH,
    resumes_path=RESUMES_PATH,
)

train_dataset = train_dataset.select(range(4))

In [ ]:
%%time

OUTPUT_PATH = "/content/drive/MyDrive/auto-hh/models/BiEncoder"

# Запускаем обучение
print("🚀 Начинаем обучение...")

trainer = Trainer(
    batch_size=2,
    epochs=1,
)
trainer.train(model, train_dataset, OUTPUT_PATH)

print("=" * 50)
print("✅ Обучение завершено!")
print(f"📁 Модель сохранена в: {OUTPUT_PATH}")

In [ ]:
import torch, gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

## 🧪 ПРОВЕРКА ЗАГРУЗКИ И РАБОТЫ БИ ЭНКОДЕРА

In [ ]:
print("\n📥 Загрузка модели...")
try:
    loaded_model = BiEncoder.load_trained(
        path=OUTPUT_PATH,
        model_name="BAAI/bge-m3",
        need_attention=True
    )
    loaded_model.eval()

    if torch.cuda.is_available():
        loaded_model = loaded_model.to('cuda')

    print(f"✅ Модель загружена из: {OUTPUT_PATH}")
    print(f"   Устройство: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"   Размерность эмбеддинга: {loaded_model.get_sentence_embedding_dimension()}")
except Exception as e:
    print(f"❌ Ошибка загрузки: {e}")
    raise

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

print("\n📝 Тестовые тексты:")

test_resume = """
Python разработчик с опытом 5 лет.
Стек: Django, FastAPI, PostgreSQL, Docker, Kubernetes.
Опыт работы с микросервисами и Kafka.
"""

test_vacancy_match = """
Требуется Senior Python разработчик.
Стек: Django, FastAPI, PostgreSQL.
Опыт с микросервисами и очередями сообщений.
"""

test_vacancy_mismatch = """
Требуется Java разработчик.
Стек: Spring, Hibernate, MySQL.
Опыт работы с корпоративными системами.
"""

print(f"   Резюме: {test_resume[:50]}...")
print(f"   Вакансия (релевантная): {test_vacancy_match[:50]}...")
print(f"   Вакансия (нерелевантная): {test_vacancy_mismatch[:50]}...")

# ─── 3. Векторизация ─────────────────────
print("\n🔢 Векторизация...")

emb_resume = loaded_model.encode(test_resume, convert_to_tensor=True)
emb_match = loaded_model.encode(test_vacancy_match, convert_to_tensor=True)
emb_mismatch = loaded_model.encode(test_vacancy_mismatch, convert_to_tensor=True)

print(f"   Эмбеддинг резюме: {emb_resume.shape}")
print(f"   Эмбеддинг вакансии (match): {emb_match.shape}")
print(f"   Эмбеддинг вакансии (mismatch): {emb_mismatch.shape}")

# ─── 4. Косинусное сходство ──────────────
print("\n📊 Косинусное сходство:")

sim_match = util.cos_sim(emb_resume, emb_match).item()
sim_mismatch = util.cos_sim(emb_resume, emb_mismatch).item()

print(f"   Резюме ↔ Релевантная вакансия: {sim_match:.4f}")
print(f"   Резюме ↔ Нерелевантная вакансия: {sim_mismatch:.4f}")
print(f"   Разница: {sim_match - sim_mismatch:.4f}")

# ─── 5. Валидация ────────────────────────
print("\n✅ ВАЛИДАЦИЯ:")

if sim_match > sim_mismatch:
    print("   ✅ Модель работает корректно!")
    print("   ✅ Релевантная вакансия имеет больший скор")
else:
    print("   ⚠️ Модель требует дообучения")
    print("   ⚠️ Нерелевантная вакансия имеет больший скор")

if sim_match > 0.5:
    print("   ✅ Хорошее сходство для релевантной пары")
else:
    print("   ⚠️ Низкое сходство (возможно, мало данных для обучения)")

# ─── 6. Тест скорости ────────────────────
print("\n⚡ ТЕСТ СКОРОСТИ:")

import time

n_requests = 10
start = time.time()
for _ in range(n_requests):
    loaded_model.encode([test_resume], convert_to_numpy=True)
end = time.time()

avg_time = (end - start) / n_requests * 1000  # мс
print(f"   Среднее время на запрос: {avg_time:.2f} мс")
print(f"   Запросов в секунду: {1000/avg_time:.1f}")

# ────────────────────────────────────────
print("\n" + "=" * 60)
print("🎉 ТЕСТИРОВАНИЕ ЗАВЕРШЕНО")
print("=" * 60)

## ТЕСТ КРОСС ЭНКОДЕРА

In [ ]:
# ─── Загрузка ──────────────────────────
print("📥 Загрузка CrossEncoder...")
ce = CrossEncoder("BAAI/bge-reranker-v2-m3")

In [15]:
query = "Python разработчик Django"
candidates = [
    "Требуется Python разработчик с опытом Django и FastAPI",
    "Вакансия: Java разработчик Spring Boot",
    "Ищем DevOps инженера Kubernetes Docker",
]

scores = ce.rerank(query, candidates)
print(*scores, sep='\n')

# ─── Проверка ──────────────────────────
assert len(scores) == 3, "❌ Количество scores не совпадает!"
assert 0 <= scores[0]["score"] <= 1, "❌ Score вне диапазона!"
print("\n✅ CrossEncoder работает! 🎉")

{'document': 'Требуется Python разработчик с опытом Django и FastAPI', 'score': 0.9904739260673523}
{'document': 'Вакансия: Java разработчик Spring Boot', 'score': 0.011138050816953182}
{'document': 'Ищем DevOps инженера Kubernetes Docker', 'score': 0.001354677020572126}

✅ CrossEncoder работает! 🎉
